In [3]:
import pandas as pd

In [4]:
results = pd.read_csv('usgsid_comid_results_2025.csv', dtype={'USGSID': str, 'COMID': str})

In [5]:
results2 = pd.read_csv('site_info_active.csv', dtype={'site_no': str})

In [6]:
print(results.head())

     USGSID     COMID
0  01154950   9331072
1  14330000  23923628
2  01672500   8507536
3  06800500  24750511
4  01127000   6149075


In [7]:
print(results2.head())

     site_no                                         station_nm  dec_lat_va  \
0   02339495                      OSELIGEE CREEK NEAR LANETT AL   32.901516   
1   02342500                UCHEE CREEK NEAR FORT MITCHELL, AL.   32.316815   
2   02342937               SOUTH FORK COWIKEE CREEK NR HOWE, AL   32.016786   
3  023432415  CHATTAHOOCHEE R .36 MI DS WFG DAM NR FT GAINES...   31.622194   
4   02361000              CHOCTAWHATCHEE RIVER NEAR NEWTON, AL.   31.342949   

   dec_long_va  alt_va  parm_cd  
0   -85.196331     NaN       60  
1   -85.014931  201.63       60  
2   -85.213800  196.00       60  
3   -85.059167    0.01       60  
4   -85.610491  138.56       60  


In [8]:
import geopandas as gpd

sites_gdf = gpd.GeoDataFrame(
    results2,
    geometry=gpd.points_from_xy(results2['dec_long_va'],results2['dec_lat_va']),
    crs="EPSG:4326"
)

In [9]:
print(sites_gdf.head())

     site_no                                         station_nm  dec_lat_va  \
0   02339495                      OSELIGEE CREEK NEAR LANETT AL   32.901516   
1   02342500                UCHEE CREEK NEAR FORT MITCHELL, AL.   32.316815   
2   02342937               SOUTH FORK COWIKEE CREEK NR HOWE, AL   32.016786   
3  023432415  CHATTAHOOCHEE R .36 MI DS WFG DAM NR FT GAINES...   31.622194   
4   02361000              CHOCTAWHATCHEE RIVER NEAR NEWTON, AL.   31.342949   

   dec_long_va  alt_va  parm_cd                    geometry  
0   -85.196331     NaN       60  POINT (-85.19633 32.90152)  
1   -85.014931  201.63       60  POINT (-85.01493 32.31681)  
2   -85.213800  196.00       60   POINT (-85.2138 32.01679)  
3   -85.059167    0.01       60  POINT (-85.05917 31.62219)  
4   -85.610491  138.56       60  POINT (-85.61049 31.34295)  


In [10]:
print(results['USGSID'].str.len().value_counts())
print(sites_gdf['site_no'].str.len().value_counts())

USGSID
8     7440
9      118
15      85
10       1
Name: count, dtype: int64
site_no
8     9072
10     130
15     109
9      108
Name: count, dtype: int64


In [11]:
sites_gdf.rename(columns={'site_no': "USGSID"}, inplace=True)

In [12]:
print(sites_gdf.head())

      USGSID                                         station_nm  dec_lat_va  \
0   02339495                      OSELIGEE CREEK NEAR LANETT AL   32.901516   
1   02342500                UCHEE CREEK NEAR FORT MITCHELL, AL.   32.316815   
2   02342937               SOUTH FORK COWIKEE CREEK NR HOWE, AL   32.016786   
3  023432415  CHATTAHOOCHEE R .36 MI DS WFG DAM NR FT GAINES...   31.622194   
4   02361000              CHOCTAWHATCHEE RIVER NEAR NEWTON, AL.   31.342949   

   dec_long_va  alt_va  parm_cd                    geometry  
0   -85.196331     NaN       60  POINT (-85.19633 32.90152)  
1   -85.014931  201.63       60  POINT (-85.01493 32.31681)  
2   -85.213800  196.00       60   POINT (-85.2138 32.01679)  
3   -85.059167    0.01       60  POINT (-85.05917 31.62219)  
4   -85.610491  138.56       60  POINT (-85.61049 31.34295)  


In [13]:
df_merged = pd.merge(sites_gdf, results, on='USGSID', how='left')

In [14]:
print(df_merged.tail())

        USGSID                                         station_nm  dec_lat_va  \
9414  16847000                        Imong River near Agat, Guam   13.339000   
9415  16848100                     Almagosa River near Agat, Guam   13.346722   
9416  16848500                       Maulap River near Agat, Guam   13.355444   
9417  16854500  Ugum River above Talofofo Falls, nr Talofofo, ...   13.322417   
9418  16865000                        Pago River near Ordot, Guam   13.436806   

      dec_long_va  alt_va  parm_cd                    geometry COMID  
9414   144.701528   120.0       60    POINT (144.70153 13.339)   NaN  
9415   144.695500   155.0       60   POINT (144.6955 13.34672)   NaN  
9416   144.697944   130.0       60  POINT (144.69794 13.35544)   NaN  
9417   144.736139   130.0       60  POINT (144.73614 13.32242)   NaN  
9418   144.756028    23.0       60  POINT (144.75603 13.43681)   NaN  


In [15]:
df_merged.shape

(9419, 8)

In [16]:
df_merged['COMID'].isna().sum()

np.int64(1966)

In [17]:
df_merged = df_merged.drop(columns=['dec_lat_va', 'dec_long_va', 'alt_va', 'parm_cd'])

In [18]:
print(df_merged.columns)

Index(['USGSID', 'station_nm', 'geometry', 'COMID'], dtype='object')


In [19]:
# Keep only CONUS gages — drop Alaska, Hawaii, and Pacific/Caribbean territories.
# .cx is GeoPandas' bounding-box slice: df.cx[xmin:xmax, ymin:ymax] = [West:East, South:North].
# This box matches the app's map extent, so data scope and view scope stay consistent.
before = len(df_merged)
df_merged = df_merged.cx[-125:-66, 24:50]
print(f"CONUS filter: {before} -> {len(df_merged)} ({before - len(df_merged)} dropped)")


CONUS filter: 9419 -> 9113 (306 dropped)


In [20]:
# Store the prefixed USGS id form per Sudip's schema decision: "USGS-01646500".
# The guard prevents double-prefixing if the cell is re-run without reloading data.
mask = ~df_merged['USGSID'].astype(str).str.startswith('USGS-')
df_merged.loc[mask, 'USGSID'] = 'USGS-' + df_merged.loc[mask, 'USGSID'].astype(str)


In [21]:
df_merged.to_file('merged_gages.geojson', driver='GeoJSON')